# Probe Prefill Steps Visualization
Load `results/probe.json` and visualize per-step summary + step-layer heatmap.

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

probe_path = Path('results/probe.json')
data = json.loads(probe_path.read_text(encoding='utf-8'))

if 'runs' in data:
    run = data['runs'][0]
    print(f"n_prompts={data['n_prompts']}, using run[0]")
else:
    run = data

timeline = run.get('timeline') or []
if timeline:
    steps = [s for s in timeline if s.get('phase') == 'prefill'] or timeline
    step_source = 'timeline(prefill)'
else:
    steps = run.get('steps', [])
    step_source = 'steps'

print('steps:', len(steps), '| source:', step_source)
print('geometry_space:', run.get('geometry_space'))



In [ ]:
if not steps:
    raise RuntimeError('No steps found. Re-run scripts/run_probe.py after the code update.')

x = np.arange(len(steps))
dz_para = np.array([s['summary'].get('dz_para_mean', np.nan) for s in steps], dtype=float)
dz_perp = np.array([s['summary'].get('dz_perp_mean', np.nan) for s in steps], dtype=float)
io_cos = np.array([s['summary'].get('io_cos_sim_mean', np.nan) for s in steps], dtype=float)
ratio = np.array([s['summary'].get('dz_para_dz_perp_ratio_mean', np.nan) for s in steps], dtype=float)

plt.figure(figsize=(12, 4))
plt.plot(x, dz_para, label='dz_para_mean')
plt.plot(x, dz_perp, label='dz_perp_mean')
plt.plot(x, io_cos, label='io_cos_sim_mean')
plt.plot(x, ratio, label='dz_para_dz_perp_ratio_mean')
plt.xlabel('prefill step (token index)')
plt.ylabel('value')
plt.title('Prefill Step Curves')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
has_layer = all('layer_metrics' in s for s in steps)
if not has_layer:
    raise RuntimeError('No layer_metrics in steps. Run probe with print_per_layer enabled (default in scripts/run_probe.py).')

n_steps = len(steps)
n_layers = len(steps[0]['layer_metrics'])
mat = np.full((n_steps, n_layers), np.nan, dtype=float)
for i, s in enumerate(steps):
    for m in s['layer_metrics']:
        l = int(m['layer'])
        v = m.get('para_perp_ratio', None)
        if v is not None:
            mat[i, l] = float(v)

plt.figure(figsize=(12, 5))
im = plt.imshow(mat, aspect='auto', interpolation='nearest')
plt.colorbar(im, label='para_perp_ratio')
plt.xlabel('layer')
plt.ylabel('prefill step (token index)')
plt.title('Probe: para_perp_ratio (step x layer)')
plt.tight_layout()
plt.show()